# Portfolio Tear Sheet

Build a small multi-instrument, multi-entity portfolio, value it, aggregate metrics
and cashflows, then render a `portfolio_tearsheet`.

In [ ]:
import datetime as dt
import json
from datetime import date

from finstack_quant import reporting
from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.portfolio import aggregate_full_cashflows, aggregate_metrics, value_portfolio

# ----- Market context -----
mc = MarketContext()
mc.insert(DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [(0.0, 1.0), (0.25, 0.9875), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75)],
    day_count="act_365f",
))
market_json = mc.to_json()
print("Market context built; curves:", [c["id"] for c in json.loads(market_json)["curves"]])


## Portfolio spec — four deposits across two entities

In [ ]:
def dep_pos(pos_id, entity_id, instr_id, notional, rate, maturity):
    return {
        "position_id": pos_id,
        "entity_id": entity_id,
        "instrument_id": instr_id,
        "instrument_spec": {
            "type": "deposit",
            "spec": {
                "id": instr_id,
                "notional": {"amount": notional, "currency": "USD"},
                "start_date": "2025-01-15",
                "maturity": maturity,
                "day_count": "Act360",
                "quote_rate": rate,
                "discount_curve_id": "USD-OIS",
                "attributes": {},
            },
        },
        "quantity": 1.0,
        "unit": "units",
    }

portfolio_spec = {
    "id": "multi-asset-book",
    "as_of": "2025-01-15",
    "base_ccy": "USD",
    "entities": {
        "FUND-1": {"id": "FUND-1"},
        "FUND-2": {"id": "FUND-2"},
    },
    "positions": [
        dep_pos("F1-3M",  "FUND-1", "DEP-3M",  5_000_000.0, 0.045, "2025-04-15"),
        dep_pos("F1-6M",  "FUND-1", "DEP-6M",  8_000_000.0, 0.050, "2025-07-15"),
        dep_pos("F1-1Y",  "FUND-1", "DEP-1Y",  3_000_000.0, 0.055, "2026-01-15"),
        dep_pos("F2-3M",  "FUND-2", "DEP-3M-B", 4_000_000.0, 0.042, "2025-04-15"),
        dep_pos("F2-6M",  "FUND-2", "DEP-6M-B", 6_000_000.0, 0.048, "2025-07-15"),
        dep_pos("F2-2Y",  "FUND-2", "DEP-2Y",  2_000_000.0, 0.058, "2027-01-15"),
    ],
}
portfolio_json = json.dumps(portfolio_spec)
print(f"Portfolio: {len(portfolio_spec["positions"])} positions across {len(portfolio_spec["entities"])} entities")


## Valuation, metrics, and cashflows

In [ ]:
valuation = value_portfolio(portfolio_json, market_json)
metrics = aggregate_metrics(valuation, "USD", market_json, "2025-01-15")
cf_obj = aggregate_full_cashflows(portfolio_json, market_json)
cashflows = cf_obj.to_json()

val_parsed = json.loads(valuation)
total_pv = float(val_parsed["total_base_ccy"]["amount"])
print(f"Total PV: {total_pv:,.2f} USD")
print(f"Entities: {list(val_parsed["by_entity"].keys())}")


## Book summary

In [ ]:
reporting.portfolio_tearsheet(
    valuation,
    metrics=metrics,
    cashflows=cashflows,
    title="Multi-Asset Book",
    generated=dt.date(2026, 6, 23),
)

## Saving a standalone HTML file

```python
ts = reporting.portfolio_tearsheet(valuation, metrics=metrics, cashflows=cashflows, generated=dt.date(2026, 6, 23))
ts.save("portfolio_tearsheet.html")
```